In [15]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import sqlite3
from datetime import datetime

In [16]:
client = OpenAI(
    api_key="sk-GDsDnrnoU68uCBi2jhfC0A",
    base_url="https://elmodels.ngrok.app/v1"
)

In [17]:
model = SentenceTransformer("intfloat/multilingual-e5-small")

index = faiss.read_index("data/faiss_index.bin")
with open("data/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

In [18]:
def has_previous_records(cursor, patient_id):
    cursor.execute("SELECT COUNT(*) FROM visits WHERE patient_id = ?", (patient_id,))
    count = cursor.fetchone()[0]
    return count > 0

In [19]:
def register_visit(cursor, db, patient_id, hospital_name, hospital_city):
    now = datetime.now()
    visit_date = now.strftime("%Y-%m-%d")
    visit_time = now.strftime("%H:%M:%S")

    cursor.execute("""
    INSERT INTO visits (patient_id, hospital_name, hospital_city, visit_date, visit_time)
    VALUES (?, ?, ?, ?, ?)
    """, (patient_id, hospital_name, hospital_city, visit_date, visit_time))

    db.commit()

    print("تم تسجيل حضور الموعد")

In [20]:
# للم يعطي احتماليه انه الوقت غير منطقي
# rows[-1] == اخر موعد (الموعد الحالي)
# rows[:-1] == المواعيد السابقه

def analyze_visits(client, cursor, patient_id):
    cursor.execute("""
    SELECT * FROM visits
    WHERE patient_id = ?
    AND date(visit_date) BETWEEN date('now', '-7 days') AND date('now')               
    ORDER BY visit_date, visit_time
    """, (patient_id,))
    v_rows = cursor.fetchall()

    response = client.chat.completions.create(
        model="nuha-2.0",
        messages = [
            {
                "role": "user",
                "content": f"""
انت مكتشف احتيال استخدام هوية شخص اخر في المستشفى حيث يمكنك كشف هذا الاحتيال بناء على الزمن بين الموعد الحالي واي موعد سابق والاخذ بالحسبان المسافه بين المستشفيين

عليك اعطاء نسبه للموثوقية من 0-100، حيث ان 100 تعني موثوق تماماً و 0 تعني غير موثوق ابداً(منتحل)

الموعد الحالي: {v_rows[-1]}
المواعيد السابقة: {v_rows[:-1]}

ملاحظات مهمة:
ارجع فقط نسبة الموثوقية لكل موعد سابق على حده مع ذكر سبب مختصر لهذا التقييم
لا تفترض اخطاء في الادخال
لا ترجع اي كلام اضافي 

Return your answer in JSON format only.:
{{
    "الموعد_101": {{
        "النسبة": number (0 to 100),
        "السبب": ["short reason"]
    }},
    "الموعد_102": {{
        "النسبة": number (0 to 100),
        "السبب": ["short reason"]
    }},
}}

"""
            }
        ]
    )

    analyze_visits = response.choices[0].message.content
    return analyze_visits

In [21]:
def add_vitals(cursor, db, patient_id, visit_id, gender, height, weight):
    cursor.execute("""
    INSERT INTO vital_signs (patient_id, visit_id, entered_gender, height_cm, weight_kg, recorded_date)
    VALUES (?, ?, ?, ?, ?, DATE('now'))
    """, (patient_id, visit_id, gender, height, weight))

    db.commit()

In [22]:
# اكتشاف الاحتيال من خلال الطول والوزن والجنس

def analyze_vitals(client, cursor, patient_id):
    cursor.execute("""
    SELECT 
        vs.entered_gender,
        vs.height_cm,
        vs.weight_kg,
        vs.recorded_date
    FROM vital_signs vs
    WHERE patient_id = ?
    ORDER BY recorded_date DESC
    LIMIT 2
    """, (patient_id,))
    vs_rows = cursor.fetchall()

    if vs_rows[0][0] != vs_rows[1][0]:
        gender = False
    else:
        gender = True

    response = client.chat.completions.create(
        model="nuha-2.0",
        messages = [
            {
                "role": "user",
                "content": f"""
انت مكتشف احتيال استخدام هوية شخص اخر في المستشفى حيث يمكنك كشف هذا الاحتيال بناء على تقييم الاختلاف بين كشفين للطول والوزن بناء على المدة بينمها 
عليك اعطاء نسبه للموثوقية من 0-100، حيث ان 100 تعني موثوق تماماً و 0 تعني غير موثوق ابداً(منتحل)

البيانات: {vs_rows}

ملاحظات مهمة:
ارجع فقط نسبة الموثوقية لكل من الطول والوزن على حده مع ذكر سبب هذا التقييم
لا تفترض اخطاء في الادخال او القياس
لا ترجع اي كلام اضافي 

Return your answer in JSON format only:
{{
    "الطول": {{
        "النسبة": number (0 to 100),
        "السبب": ["short reason"]
    }},
    "الوزن": {{
        "النسبة": number (0 to 100),
        "السبب": ["short reason"]
    }},
}}
"""
            }
        ]
    )

    analyze_vitals = response.choices[0].message.content
    return gender, analyze_vitals

In [23]:
def add_diagnosis(cursor, db, patient_id, visit_id, diagnosis):
    cursor.execute("""
    INSERT INTO diagnoses (patient_id, visit_id, diagnosis)
    VALUES (?, ?, ?)
    """, (patient_id, visit_id, diagnosis))

    db.commit()

In [24]:
def analyze_diagnosis(client, cursor, patient_id):
    # استخراج جميع امراض المريض
    cursor.execute("""
    SELECT
        diagnosis
    FROM diagnoses
    WHERE patient_id = ?
    """, (patient_id,))
    d_rows = cursor.fetchall()

    # استحراج جنس، طول، ووزن المريض
    cursor.execute("""
    SELECT
        vs.entered_gender,
        vs.height_cm,
        vs.weight_kg,
        p.birth_year
    FROM visits v
    JOIN vital_signs vs ON v.visit_id = vs.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    WHERE v.patient_id = ?
    ORDER BY v.visit_date DESC, v.visit_time DESC
    LIMIT 1
    """, (patient_id,))
    all_rows = cursor.fetchall()

    # استخراج عمر المريض
    from datetime import date
    def calculate_age(born):
        today = date.today()
        return today.year - born
    age = calculate_age(all_rows[0][3])

    # بحث باستخدام RAG 
    def search(query, k=5):
        q = f"query: {query}"
        q_embedding = model.encode([q])
        D, I = index.search(np.array(q_embedding), k)
        results = [all_chunks[i] for i in I[0]]
        return results
    rag_results = search(d_rows)

    # llm اكتشاف الاحتيال من خلال المرض
    response = client.chat.completions.create(
        model="nuha-2.0",
        messages = [
            {
                "role": "user",
                "content": f"""
 انت مكتشف احتيال استخدام هوية شخص اخر في المستشفى حيث يمكنك كشف هذا الاحتيال بناء على تقييم امكانية الاصابة بالمرض الحالي بناء على سجل الامراض السابقة وبيانات المريض واستنادا على المعلومات الطبية  
عليك اعطاء نسبه للموثوقية من 0-100، حيث ان 100 تعني موثوق تماماً و 0 تعني غير موثوق ابداً(منتحل)

البيانات: 
المرض الحالي: {d_rows[-1]}
الامراض السابقة: {d_rows[:-1]}
بيانات المريض: {all_rows[0][:3]}
عمر المريض: {age}
بيانات من قاعدة معلومات طبية: {rag_results}


ملاحظات مهمة:
ارجع فقط نسبة الموثوقية مع ذكر سبب هذا التقييم
لا تفترض اخطاء في الادخال او القياس
لا ترجع اي كلام اضافي 

Return your answer in JSON format only:
{{
    "النسبة": number (0 to 100),
    "السبب": ["short reason"]
}}

"""
            }
        ]
    )

    analyze_diagnosis = response.choices[0].message.content
    return analyze_diagnosis

In [25]:
def get_visit_id(cursor, patient_id):
    cursor.execute("""
    SELECT visit_id FROM visits
    WHERE patient_id = ?
    ORDER BY visit_id DESC
    LIMIT 1
    """, (patient_id,))

    visit_id = cursor.fetchone()[0]
    return visit_id

In [26]:
import json

In [27]:
def calculate_final_risk(visits, g, vitals, diagnosis):
    weights = {
        "visits": 0.3,
        "g": 0.3,
        "vitals": 0.2,
        "diagnosis": 0.2
    }
    visits = json.loads(visits)
    vitals = json.loads(vitals)
    diagnosis = json.loads(diagnosis)

    total_risk = 0
    all_reasons = []

    # --- المواعيد ---
    visit_scores = []
    for v in visits.values():
        score = v["النسبة"]
        risk = 1 - (score / 100)
        visit_scores.append(risk)

        # خذ فقط الأسباب المشبوهة
        if score < 70:
            all_reasons.extend(v["السبب"])

    visits_risk = sum(visit_scores) / len(visit_scores) if visit_scores else 0
    total_risk += visits_risk * weights["visits"]

    # --- العلامات الحيوية ---
    if not g:
        total_risk += 1 * weights["g"]
        all_reasons.append("اختلاف جنس المراجع عن السجلات")

    vitals_scores = []
    for v in vitals.values():
        score = v["النسبة"]
        risk = 1 - (score / 100)
        vitals_scores.append(risk)

        if score < 90:
            all_reasons.extend(v["السبب"])

    vitals_risk = sum(vitals_scores) / len(vitals_scores) if vitals_scores else 0
    total_risk += vitals_risk * weights["vitals"]

    # --- التشخيص ---
    d_score = diagnosis["النسبة"]
    d_risk = 1 - (d_score / 100)

    if d_score < 90:
        all_reasons.extend(diagnosis["السبب"])

    total_risk += d_risk * weights["diagnosis"]

    return {
        "risk_score": total_risk * 100,
        "reasons": all_reasons
    }

In [ ]:
if __name__ == "__main__":
    db = sqlite3.connect("data/rasid.db")
    cursor = db.cursor()

    print("--- أهلا بك في راصد ---")

    patient_id = input("أدخل رقم المريض: ")
    hospital_name = input("أدخل اسم المستشفى: ")
    hospital_city = input("أدخل اسم المدينة: ")

    has_history = has_previous_records(cursor, patient_id)

    register_visit(cursor, db, patient_id, hospital_name, hospital_city)
    visit_id = get_visit_id(cursor, patient_id)
    
    if has_history:
        print("يتم تحليل المواعيد ...")
        v_result = analyze_visits(client, cursor, patient_id)
    
    gender = input("الجنس: ")
    height = float(input("الطول (سم): "))
    weight = float(input("الوزن (كجم): "))
    add_vitals(cursor, db, patient_id, visit_id, gender, height, weight)

    if has_history:
        print("يتم تحليل العلامات الحيوية ...")
        g , vs_result = analyze_vitals(client, cursor, patient_id)

    diagnosis = input("أدخل التشخيص: ")
    add_diagnosis(cursor, db, patient_id, visit_id, diagnosis)

    
    print("يتم تحليل التشخيص ...")
    d_result = analyze_diagnosis(client, cursor, patient_id)

    result = calculate_final_risk(v_result, g, vs_result, d_result)
    print(result["risk_score"])
    print(result["reasons"])

--- أهلا بك في راصد ---
تم تسجيل حضور الموعد

 تحليل المواعيد:

 إدخال العلامات الحيوية

 تحليل العلامات الحيوية:

 إدخال التشخيص

 تحليل التشخيص:
21.499999999999996
['تداخل زمني مستحيل (موعد سابق بعد الموعد الحالي)', 'تداخل زمني مستحيل (موعد سابق بعد الموعد الحالي)']

 انتهت رحلة المريض بنجاح
